In [1]:
%load_ext dotenv
%dotenv

In [2]:
from pathlib import Path
OUTPUT_ROOT_PATH = Path("../../../data/gsm-symbolic/experiments").resolve()
  
print(OUTPUT_ROOT_PATH)

/home/guests2/dkd/data/gsm-symbolic/experiments


In [3]:
import numpy as np
import logging
from huggingface_hub import login, whoami
import os

from transformers.utils.logging import disable_progress_bar
disable_progress_bar()  # error/hung up loading tokenizer otherwise

from gsm_benchmarker.dataset_wrapper import GSMSymbolicDataset
from gsm_benchmarker.benchmark_config import BenchmarkConfig
from gsm_benchmarker.models_config_parser import ModelsConfig
from gsm_benchmarker.model_evaluator import ModelEvaluator
from gsm_benchmarker.utils.logging_setup import install_colored_logger

api_token = os.environ['HUGGINGFACEHUB_API_TOKEN']

login(api_token)

whoami()['name']


openai package not installed; OpenAI models will not be available
anthropic package not installed; Anthropic models will not be available
google-genai package not installed; Gemini family models will not be available


'the-mysh'

In [4]:
for log_name in ('urllib3', 'fsspec', 'filelock', 'h5py'):
    logging.getLogger(log_name).setLevel(logging.INFO)

logger = logging.getLogger(__name__)
install_colored_logger(level=logging.DEBUG)

In [5]:
# MODEL_NAME = "microsoft/Phi-3-mini-128k-instruct"
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"

In [33]:
dataset_wrapper_main = GSMSymbolicDataset(variant=GSMSymbolicDataset.Variant.main)
dataset_wrapper_p1 = GSMSymbolicDataset(variant=GSMSymbolicDataset.Variant.p1)

gsm_benchmarker.dataset_wrapper [DEBUG] 2025-11-05 14:30:25: Loading GSM-Symbolic dataset (variant: main)...
gsm_benchmarker.dataset_wrapper [DEBUG] 2025-11-05 14:30:27: Loaded 5000 examples
gsm_benchmarker.dataset_wrapper [DEBUG] 2025-11-05 14:30:27: Loading GSM-Symbolic dataset (variant: p1)...
gsm_benchmarker.dataset_wrapper [DEBUG] 2025-11-05 14:30:28: Loaded 5000 examples


In [34]:
eval_sets_main = dataset_wrapper_main.create_evaluation_sets(n_sets=2, n_per_set=3)  # small for testing
eval_sets_p1 = dataset_wrapper_p1.create_evaluation_sets(n_sets=2, n_per_set=3)

gsm_benchmarker.dataset_wrapper [DEBUG] 2025-11-05 14:30:28: Creating 2 set(s) with 3 example(s) each
gsm_benchmarker.dataset_wrapper [DEBUG] 2025-11-05 14:30:28: Creating 2 set(s) with 3 example(s) each


In [27]:
s = eval_sets_main[0][0]
print(type(s))
print(f"{s.id}: {s.question}\n")

<class 'gsm_benchmarker.dataset_wrapper.GSMSymbolicDataset.Sample'>
81: The Charity fundraiser team hid 180 coins. The Lopez twins each found 43 coins. All the other coins except 17 were found by their friends. How many coins did the friends find?



In [11]:
# test on a small model
config = BenchmarkConfig(gpu0_max_memory="30GiB", cpu_max_memory="60GiB")
models_config = ModelsConfig()
evaluator = ModelEvaluator(models_config[MODEL_NAME], config)

gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:18:17: Initialising a HuggingFace model
gsm_benchmarker.hf_model_wrapper [INFO] 2025-11-05 14:18:17: Setting up model meta-llama/Meta-Llama-3-8B-Instruct
gsm_benchmarker.hf_model_wrapper [DEBUG] 2025-11-05 14:18:17: Loading tokeniser
gsm_benchmarker.hf_model_wrapper [DEBUG] 2025-11-05 14:18:18: CUDA available
gsm_benchmarker.hf_model_wrapper [DEBUG] 2025-11-05 14:18:18: Loading model with CUDA
bitsandbytes.cextension [DEBUG] 2025-11-05 14:18:19: Loading bitsandbytes native library from: /home/guests2/dkd/code/gsm-symbolic-benchmarking/.venv/lib/python3.12/site-packages/bitsandbytes/libbitsandbytes_cuda126.so
gsm_benchmarker.hf_model_wrapper [INFO] 2025-11-05 14:19:04: Model loaded


In [28]:
prompt = evaluator.create_prompt(eval_sets_main[0][0].question)
print(prompt)

As an expert problem solver, solve step by step the following mathematical questions.

Q: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
A: Let's think step by step. There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The final answer is 6.

Q: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
A: Let's think step by step. There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The final answer is 5.

Q: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
A: Let's think step by step. Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. The final answer is 39.

Q: Jason had 20 lollipops. He gave Denny some

In [29]:
resp = evaluator.model_wrapper.ask(prompt)
print(resp)

/home/guests2/dkd/code/gsm-symbolic-benchmarking/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


 There were originally 180 coins. The twins found 43 each, so that is 43 + 43 = 86 coins. The friends found all the coins except 17, so they found 180 - 17 = 163 coins. The friends found 163 - 86 = 77 coins. The final answer is 77.


In [14]:
prompt = evaluator.create_prompt("The Treasure hunt team hid 100 toys. The Lee twins each found 13 toys. All the remaining toys except 5 were found by their friends. What percentage of the initially hidden toys did the friends find?")
resp = evaluator.model_wrapper.ask(prompt)
print(resp)

 The team hid 100 toys. The Lee twins found 13 each, so that is 13 + 13 = 26 toys. 100 - 26 = 74 toys were left. The friends found all except 5, so they found 74 - 5 = 69 toys. The percentage of the initially hidden toys that the friends found is 69/100 = 69%. The final answer is 69%.


In [35]:
results_main, _ = evaluator.evaluate_multiple_datasets(eval_sets_main)
results_main

gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:30:35: No intermediate storage path provided; intermediate results will not be stored


Dataset:   0%|          | 0/2 [00:00<?, ?it/s]

Example:   0%|          | 0/3 [00:00<?, ?it/s]

gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:30:35: Matched answer pattern: GSM8K standard format: '#### <number>
/home/guests2/dkd/code/gsm-symbolic-benchmarking/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:30:44: Matched answer pattern: GSM-Symbolic format: 'The final answer is <number>'
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:30:44: Matched answer pattern: GSM8K standard format: '#### <number>
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:30:58: Matched answer pattern: GSM-Symbolic format: 'The final answer is <number>'
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:30:58: No predefined answer pattern matched
gsm_benchmarker.model_evaluator [DEBUG] 20

Example:   0%|          | 0/3 [00:00<?, ?it/s]

gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:31:07: Matched answer pattern: GSM8K standard format: '#### <number>
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:31:14: Matched answer pattern: GSM-Symbolic format: 'The final answer is <number>'
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:31:14: Matched answer pattern: GSM8K standard format: '#### <number>
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:31:22: Matched answer pattern: GSM-Symbolic format: 'The final answer is <number>'
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:31:22: No predefined answer pattern matched
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:31:22: Extracting answer as the last number in text
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:31:32: Matched answer pattern: GSM-Symbolic format: 'The final answer is <number>'


id  original_id  \
set_number question_number                    
0          0                81         1025   
           1                58         1026   
           2                46         1031   
1          0                81         1025   
           1                58         1026   
           2                46         1031   

                                                                     question  \
set_number question_number                                                      
0          0                The Charity fundraiser team hid 180 coins. The...   
           1                One troll can mine 5 kgs of ore per day with b...   
           2                Oakwood High School is holding graduation in t...   
1          0                The Treasure hunt team hid 200 balls. The Lope...   
           1                One troll can mine 5 kgs of ore per day with w...   
           2                Greenville High School is holding graduation i...   

                            true_answer  predicted_answer  correct  \
set_number question_number                                           
0          0                        7.0              77.0    False   
           1                      992.0            9920.0    False   
           2                        5.0               5.0     True   
1          0                       16.0             166.0    False   
           1                      980.0            9800.0    False   
           2                        8.0               8.0     True   

                                                                     response  
set_number question_number                                                     
0          0                 There were originally 180 coins. The twins fo...  
           1                 One troll can mine 5 kgs of ore per day with ...  
           2                 The auditorium can hold 7300 people. 1200 sea...  
1          0                 The team hid 200 balls. The twins found 11 ea...  
           1                 One troll with a titanium pickaxe can mine 2 ...  
           2                 The Convention Center can hold 8400 people. 9...

In [37]:
results_main

id  original_id  \
set_number question_number                    
0          0                81         1025   
           1                58         1026   
           2                46         1031   
1          0                81         1025   
           1                58         1026   
           2                46         1031   

                                                                     question  \
set_number question_number                                                      
0          0                The Charity fundraiser team hid 180 coins. The...   
           1                One troll can mine 5 kgs of ore per day with b...   
           2                Oakwood High School is holding graduation in t...   
1          0                The Treasure hunt team hid 200 balls. The Lope...   
           1                One troll can mine 5 kgs of ore per day with w...   
           2                Greenville High School is holding graduation i...   

                            true_answer  predicted_answer  correct  \
set_number question_number                                           
0          0                        7.0              77.0    False   
           1                      992.0            9920.0    False   
           2                        5.0               5.0     True   
1          0                       16.0             166.0    False   
           1                      980.0            9800.0    False   
           2                        8.0               8.0     True   

                                                                     response  
set_number question_number                                                     
0          0                 There were originally 180 coins. The twins fo...  
           1                 One troll can mine 5 kgs of ore per day with ...  
           2                 The auditorium can hold 7300 people. 1200 sea...  
1          0                 The team hid 200 balls. The twins found 11 ea...  
           1                 One troll with a titanium pickaxe can mine 2 ...  
           2                 The Convention Center can hold 8400 people. 9...

In [36]:
results_p1, _ = evaluator.evaluate_multiple_datasets(eval_sets_p1)
results_p1


gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:31:32: No intermediate storage path provided; intermediate results will not be stored


Dataset:   0%|          | 0/2 [00:00<?, ?it/s]

Example:   0%|          | 0/3 [00:00<?, ?it/s]

gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:31:32: Matched answer pattern: GSM8K standard format: '#### <number>
/home/guests2/dkd/code/gsm-symbolic-benchmarking/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:31:42: Matched answer pattern: GSM-Symbolic format: 'The final answer is <number>'
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:31:42: Matched answer pattern: GSM8K standard format: '#### <number>
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:32:04: Matched answer pattern: GSM-Symbolic format: 'The final answer is <number>'
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:32:04: Matched answer pattern: '= <number> format'
gsm_benchmarker.model_evaluator [DE

Example:   0%|          | 0/3 [00:00<?, ?it/s]

gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:32:12: Matched answer pattern: GSM8K standard format: '#### <number>
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:32:21: Matched answer pattern: GSM-Symbolic format: 'The final answer is <number>'
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:32:21: Matched answer pattern: GSM8K standard format: '#### <number>
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:32:37: Matched answer pattern: GSM-Symbolic format: 'The final answer is <number>'
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:32:37: Matched answer pattern: '= <number> format'
gsm_benchmarker.model_evaluator [DEBUG] 2025-11-05 14:32:49: Matched answer pattern: GSM-Symbolic format: 'The final answer is <number>'


id  original_id  \
set_number question_number                    
0          0                81         1025   
           1                58         1026   
           2                46         1031   
1          0                81         1025   
           1                58         1026   
           2                46         1031   

                                                                     question  \
set_number question_number                                                      
0          0                The Treasure hunt team hid 100 toys. The Lee t...   
           1                One miner can mine 5 pounds of ore per day wit...   
           2                Greenville High School is holding graduation i...   
1          0                The Charity fundraiser team hid 50 balls. The ...   
           1                One troll can mine 7 kgs of ore per day with b...   
           2                Sunnyside High is holding graduation in its Sp...   

                            true_answer  predicted_answer  correct  \
set_number question_number                                           
0          0                        6.0              69.0    False   
           1                      436.0            2688.0    False   
           2                     1700.0               2.0    False   
1          0                        3.0              30.0    False   
           1                      448.0            2688.0    False   
           2                     2250.0               3.0    False   

                                                                     response  
set_number question_number                                                     
0          0                 The team hid 100 toys. The Lee twins found 13...  
           1                 One miner can mine 5 pounds of ore per day wi...  
           2                 The Convention Center has space for 3000 peop...  
1          0                 There were originally 50 balls. The twins fou...  
           1                 One troll can mine 7 kgs of ore per day with ...  
           2                 The Sports Arena can hold 3500 people. 750 se...

In [38]:
tmp_path = Path("../../../data/gsm-symbolic/tmp").resolve()
tmp_path

PosixPath('/home/guests2/dkd/data/gsm-symbolic/tmp')

In [40]:
os.makedirs(tmp_path, exist_ok=True)
f = tmp_path/'tmp_p1.parquet'
results_p1.to_parquet(f)

In [43]:
import pandas as pd
from gsm_benchmarker.results_analyser import ModelResultsAnalyser, MultiModelResultsAnalyser, \
    MultiVariantMultiModelResultsAnalyser


In [42]:
pd.read_parquet(f)

id  original_id  \
set_number question_number                    
0          0                81         1025   
           1                58         1026   
           2                46         1031   
1          0                81         1025   
           1                58         1026   
           2                46         1031   

                                                                     question  \
set_number question_number                                                      
0          0                The Treasure hunt team hid 100 toys. The Lee t...   
           1                One miner can mine 5 pounds of ore per day wit...   
           2                Greenville High School is holding graduation i...   
1          0                The Charity fundraiser team hid 50 balls. The ...   
           1                One troll can mine 7 kgs of ore per day with b...   
           2                Sunnyside High is holding graduation in its Sp...   

                            true_answer  predicted_answer  correct  \
set_number question_number                                           
0          0                        6.0              69.0    False   
           1                      436.0            2688.0    False   
           2                     1700.0               2.0    False   
1          0                        3.0              30.0    False   
           1                      448.0            2688.0    False   
           2                     2250.0               3.0    False   

                                                                     response  
set_number question_number                                                     
0          0                 The team hid 100 toys. The Lee twins found 13...  
           1                 One miner can mine 5 pounds of ore per day wi...  
           2                 The Convention Center has space for 3000 peop...  
1          0                 There were originally 50 balls. The twins fou...  
           1                 One troll can mine 7 kgs of ore per day with ...  
           2                 The Sports Arena can hold 3500 people. 750 se...

In [45]:
mp1 = ModelResultsAnalyser(f)

In [46]:
mp1.data

id  original_id  \
set_number question_number                    
0          0                81         1025   
           1                58         1026   
           2                46         1031   
1          0                81         1025   
           1                58         1026   
           2                46         1031   

                                                                     question  \
set_number question_number                                                      
0          0                The Treasure hunt team hid 100 toys. The Lee t...   
           1                One miner can mine 5 pounds of ore per day wit...   
           2                Greenville High School is holding graduation i...   
1          0                The Charity fundraiser team hid 50 balls. The ...   
           1                One troll can mine 7 kgs of ore per day with b...   
           2                Sunnyside High is holding graduation in its Sp...   

                            true_answer  predicted_answer  correct  \
set_number question_number                                           
0          0                        6.0              69.0    False   
           1                      436.0            2688.0    False   
           2                     1700.0               2.0    False   
1          0                        3.0              30.0    False   
           1                      448.0            2688.0    False   
           2                     2250.0               3.0    False   

                                                                     response  
set_number question_number                                                     
0          0                 The team hid 100 toys. The Lee twins found 13...  
           1                 One miner can mine 5 pounds of ore per day wi...  
           2                 The Convention Center has space for 3000 peop...  
1          0                 There were originally 50 balls. The twins fou...  
           1                 One troll can mine 7 kgs of ore per day with ...  
           2                 The Sports Arena can hold 3500 people. 750 se...